# Phase 08 — BGE reranking

This notebook reranks a **real hybrid-retrieval candidate set** using the exact `BAAI/bge-reranker-v2-m3` cross-encoder. The intended pipeline is:

`Hybrid Retrieval → Top-N candidates → BGE reranker → Top-K final context`

It deliberately refuses to substitute BM25 candidates for hybrid candidates, invent reranker scores, or create a final context when Phase 07 has no real hybrid output. It does not implement LLM generation or LangGraph.

## Exact model interface

The official BGE documentation describes `BAAI/bge-reranker-v2-m3` as a multilingual 568M cross-encoder and shows `FlagReranker(...).compute_score(query-passage pairs, normalize=True)` for relevance scoring. Higher normalized scores indicate greater relevance. [BGE reranker documentation](https://bge-model.com/tutorial/5_Reranking/5.2.html) · [official model card](https://huggingface.co/BAAI/bge-reranker-v2-m3)

In [1]:
from __future__ import annotations

import importlib.metadata
import json
from itertools import islice
from pathlib import Path
from typing import Any, Iterable

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
HYBRID_STATUS_PATH = PROCESSED_DIR / 'introduction_to_business_hybrid_retrieval_status.json'
HYBRID_RESULTS_PATH = PROCESSED_DIR / 'introduction_to_business_hybrid_retrieval_results.json'
RERANK_STATUS_PATH = PROCESSED_DIR / 'introduction_to_business_reranking_status.json'
RERANKED_CONTEXTS_PATH = PROCESSED_DIR / 'introduction_to_business_reranked_contexts.json'

RERANKER_MODEL = 'BAAI/bge-reranker-v2-m3'
TOP_N_CANDIDATES = 12
TOP_K_CONTEXT = 5
RERANK_BATCH_SIZE = 4

print({'project_root': str(PROJECT_ROOT), 'top_n': TOP_N_CANDIDATES, 'top_k': TOP_K_CONTEXT, 'reranker_model': RERANKER_MODEL})

{'project_root': '/home/ubuntu/business-knowledge-ai', 'top_n': 12, 'top_k': 5, 'reranker_model': 'BAAI/bge-reranker-v2-m3'}


## Preflight: require a real hybrid candidate artifact

Phase 07 executed BM25 over real chunks but correctly blocked dense retrieval and reciprocal-rank fusion because no real BGE-M3 embeddings exist. Reranking therefore requires the Phase 07 hybrid-results artifact rather than treating sparse-only results as hybrid candidates.

In [2]:
def read_json(path: Path) -> Any:
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

hybrid_status = read_json(HYBRID_STATUS_PATH) if HYBRID_STATUS_PATH.exists() else {}
embedding_ready = bool(hybrid_status.get('embedding_artifact_exists'))
dense_ready = bool(hybrid_status.get('dense_retrieval_ready'))
hybrid_ready = hybrid_status.get('hybrid', {}).get('status') == 'completed'
candidate_artifact_exists = HYBRID_RESULTS_PATH.exists()
reranking_ready = embedding_ready and dense_ready and hybrid_ready and candidate_artifact_exists

preflight = {
    'phase': '08_reranking',
    'reranker_model': RERANKER_MODEL,
    'hybrid_status_path': str(HYBRID_STATUS_PATH.relative_to(PROJECT_ROOT)),
    'hybrid_candidates_path': str(HYBRID_RESULTS_PATH.relative_to(PROJECT_ROOT)),
    'embedding_artifact_exists': embedding_ready,
    'dense_retrieval_ready': dense_ready,
    'hybrid_retrieval_ready': hybrid_ready,
    'hybrid_candidates_exist': candidate_artifact_exists,
    'reranking_ready': reranking_ready,
    'top_n_candidates': TOP_N_CANDIDATES,
    'top_k_final_context': TOP_K_CONTEXT,
    'reranking_performed': False,
    'llm_generation_implemented': False,
    'langgraph_implemented': False,
}

if not reranking_ready:
    preflight['status'] = 'blocked_missing_real_hybrid_candidates'
    preflight['limitation'] = (
        'Phase 07 has no real BGE-M3 dense output or fused hybrid candidate artifact. '
        'No candidate ordering, BGE reranker score, or final context was fabricated.'
    )

print(json.dumps(preflight, indent=2))

{
  "phase": "08_reranking",
  "reranker_model": "BAAI/bge-reranker-v2-m3",
  "hybrid_status_path": "data/processed/introduction_to_business_hybrid_retrieval_status.json",
  "hybrid_candidates_path": "data/processed/introduction_to_business_hybrid_retrieval_results.json",
  "embedding_artifact_exists": false,
  "dense_retrieval_ready": false,
  "hybrid_retrieval_ready": false,
  "hybrid_candidates_exist": false,
  "reranking_ready": false,
  "top_n_candidates": 12,
  "top_k_final_context": 5,
  "reranking_performed": false,
  "llm_generation_implemented": false,
  "langgraph_implemented": false,
  "status": "blocked_missing_real_hybrid_candidates",
  "limitation": "Phase 07 has no real BGE-M3 dense output or fused hybrid candidate artifact. No candidate ordering, BGE reranker score, or final context was fabricated."
}


## Real reranking path (runs only after preflight passes)

The following cell loads only the exact reranker model, scores query-passage pairs in batches, preserves the full candidate metadata, and compares incoming hybrid rank with reranked rank. It uses CPU mode by default to avoid unsupported GPU assumptions; switch `devices` only after confirming an available accelerator.

In [3]:
def batched(items: list[Any], size: int) -> Iterable[list[Any]]:
    iterator = iter(items)
    while batch := list(islice(iterator, size)):
        yield batch

def normalise_runs(payload: Any) -> list[dict[str, Any]]:
    if isinstance(payload, dict):
        payload = payload.get('runs', payload.get('results', []))
    if not isinstance(payload, list):
        raise ValueError('Hybrid candidate artifact must contain a list of retrieval runs.')
    return payload

def rerank_run(reranker: Any, run: dict[str, Any]) -> dict[str, Any]:
    question = run['question']
    incoming = list(run.get('results', []))[:TOP_N_CANDIDATES]
    pairs = [[question, candidate['text']] for candidate in incoming]
    scores: list[float] = []
    for pair_batch in batched(pairs, RERANK_BATCH_SIZE):
        batch_scores = reranker.compute_score(pair_batch, normalize=True)
        if not isinstance(batch_scores, list):
            batch_scores = [batch_scores]
        scores.extend(float(score) for score in batch_scores)

    scored = []
    for hybrid_rank, (candidate, score) in enumerate(zip(incoming, scores), start=1):
        item = dict(candidate)
        item['hybrid_rank'] = hybrid_rank
        item['hybrid_score'] = candidate.get('rrf_score', candidate.get('score'))
        item['reranker_score'] = score
        scored.append(item)

    reranked = sorted(scored, key=lambda item: item['reranker_score'], reverse=True)
    for reranked_rank, item in enumerate(reranked, start=1):
        item['reranked_rank'] = reranked_rank
        item['rank_changed'] = item['hybrid_rank'] != reranked_rank

    return {
        'query_id': run.get('query_id'),
        'question': question,
        'top_n_candidates': len(incoming),
        'top_k_final_context': min(TOP_K_CONTEXT, len(reranked)),
        'before_reranking': scored,
        'after_reranking': reranked,
        'final_context': reranked[:TOP_K_CONTEXT],
    }

if reranking_ready:
    from FlagEmbedding import FlagReranker

    candidate_runs = normalise_runs(read_json(HYBRID_RESULTS_PATH))
    reranker = FlagReranker(RERANKER_MODEL, devices=['cpu'], use_fp16=False)
    reranked_contexts = [rerank_run(reranker, run) for run in candidate_runs]

    with RERANKED_CONTEXTS_PATH.open('w', encoding='utf-8') as handle:
        json.dump(reranked_contexts, handle, ensure_ascii=False, indent=2)

    preflight.update({
        'status': 'completed_with_real_hybrid_candidates',
        'reranking_performed': True,
        'reranked_contexts_path': str(RERANKED_CONTEXTS_PATH.relative_to(PROJECT_ROOT)),
        'run_count': len(reranked_contexts),
        'rank_changes_observed': sum(
            item['rank_changed'] for run in reranked_contexts for item in run['after_reranking']
        ),
    })
else:
    reranked_contexts = []
    print('Reranking not executed: real hybrid candidates are required.')

Reranking not executed: real hybrid candidates are required.


## Before-and-after ordering comparison

When real hybrid candidates exist, this display makes ordering changes auditable. A change indicates only that the cross-encoder gave a candidate a different relevance score; it is not a claim of answer quality. Review the displayed text and provenance before using the Top-K final context.

In [4]:
if reranked_contexts:
    for run in reranked_contexts[:2]:
        print(f"\nQUESTION: {run['question']}")
        print('Before → after reranking (Top-N):')
        for item in run['after_reranking']:
            chapter = item.get('chapter') or {}
            section = item.get('section') or {}
            print({
                'chunk_id': item['chunk_id'],
                'hybrid_rank': item['hybrid_rank'],
                'reranked_rank': item['reranked_rank'],
                'rank_changed': item['rank_changed'],
                'hybrid_score': item['hybrid_score'],
                'reranker_score': round(item['reranker_score'], 6),
                'page': item.get('page'),
                'chapter': chapter.get('number'),
                'section': section.get('number'),
                'text_preview': item['text'][:240].replace('\n', ' '),
            })
        print('Final Top-K context IDs:', [item['chunk_id'] for item in run['final_context']])
else:
    print('No before/after comparison is shown because no real reranking ran.')

No before/after comparison is shown because no real reranking ran.


In [5]:
try:
    preflight['flagembedding_version'] = importlib.metadata.version('FlagEmbedding')
except importlib.metadata.PackageNotFoundError:
    preflight['flagembedding_version'] = None

with RERANK_STATUS_PATH.open('w', encoding='utf-8') as handle:
    json.dump(preflight, handle, ensure_ascii=False, indent=2)

print(f'Status saved: {RERANK_STATUS_PATH.relative_to(PROJECT_ROOT)}')
print('Reranked context artifact:', RERANKED_CONTEXTS_PATH.exists())
print(json.dumps({key: preflight.get(key) for key in ['status', 'reranking_performed', 'run_count', 'rank_changes_observed', 'limitation']}, indent=2))

Status saved: data/processed/introduction_to_business_reranking_status.json
Reranked context artifact: False
{
  "status": "blocked_missing_real_hybrid_candidates",
  "reranking_performed": false,
  "run_count": null,
  "rank_changes_observed": null,
  "limitation": "Phase 07 has no real BGE-M3 dense output or fused hybrid candidate artifact. No candidate ordering, BGE reranker score, or final context was fabricated."
}


## Phase boundary

This phase ends after selecting a Top-K final context from a real hybrid candidate set. It does **not** generate an answer, call an LLM, construct prompts, or use LangGraph.